# **Port Charles stress test case - Influence of input parameters**

We are now going to toy around with some of the inputs. We will focus on the influence of
- Adjusting the simulation region
- Boundary conditions (river injections, tide...)
- Grid resolution
> See parameters section of the manual for details https://cyprienbosserelle.github.io/BG_Flood/ParametersList-py/
---


### **1. Adjusting the region of interest**
Looking closely at the h/h_P0 (timestep/Band=6), unphysical stripes are noticeable. This is caused by the domain extending beyond the range of the DEM (open `PortCharles_DEM.nc` to check)

By default, the simulation domain adjusts with the dimensions of the DEM but misalignments can appear (such as here)
To bypass this issue, the shape of the domain can be adjusted in multiple ways.

#### *1.1 Specify domain bounds*
The domain bounds can be set directly in the BG_param.txt file. 

```markdown
################
# Domain 
################
xmin = 1815750.
xmax = 1824850.
ymin = 5950380.
ymax = 5958440.
```

#### *1.2 Area of interest polygon*
The area of interest (AOI) polygon is a mask which forces BG-Flood to ignore the cells outside the polygon defined by the mask. The AOI does not modify the shape of the domain but reduces the computational load by the number of cells masked. It is usually used in conjunction with defining the domain bounds 

It must be 2 column comma-separated table *.txt file containing the x,y coordinates of the serie of points defining the polygon mask.
> Note: For the polygon to be valid, the first point must be repeated at the end.

```markdown
################
# Domain 
################
xmin = 1815750.
xmax = 1824850.
ymin = 5950380.
ymax = 5958440.

aoi = aoi_square.txt
```

***-> Try to create a simple AOI polygon and compare the simulation (runtime and output) with `aoi_watershed.txt`***

---

### **2. Boundary conditions**
4 Main types of boundary conditions are available.

```markdown
################
# Boundaries
################
# Main types of boundaries depending on the flag
# 0: Wall
# 1: Zero-gradient (Neumann): Used for open outlets
# 2: Fixed value (Dirichlet) - Requires timeseries file: Used to impose tide
# 3: Absorbing wave - Same as 2 but more stable numerically (allows waves to exit the domain)
left = 0;
right = tide.txt,2; #0;
top = tide.txt,2; #0;
bot = 0;
```

***Try to switch the top and right boundary conditions to have them vary with tide.txt. What do you notice in the simulation output?***
 - Try to check the tide file and vary the amplitude and frequency.
 - Try to increase the simulation time to see what happens?
 > Turn off the rainfall forcing to accelerate the run (Simply delete or comment the line)

 ---

### **3. Grid resolution**
The grid resolution determines how well local features are captured, at the cost of inflating the computational time.

#### 3.1 Uniform grid
To refine the simulation, decrease the value `dx`.

***Test `dx=16` and `dx=8`***
> The computational time increases because of the acrued number of cell, but also due to the numerical formulation of the time step $$\Delta t = \frac{CFL \cdot \Delta x}{u}$$


#### 3.2 Targeted refinement
To avoid drastic explosion of the computational times, local refinement can be prescribed via a *.nc file mapping the various levels of refinement.

```markdown
##########
# Grid refinement
##########

initlevel=0; # Reference refinement level
minlevel=0;  # Minimum refinement level
maxlevel=2;  # Maximum refinement level
Adaptation = Targetlevel,refinement_map.nc ;
```
***Try adding refinement using the `refinement_map.nc` already present. Does it make a difference?***
> Indeed, each variable of the simulation output are stored on separate grids corresponding to the different levels of resolution.
> The level of resolution is identified by the suffix `_P{level}`

---

### **4. Multi-resolution simulation output reconstruction**
Multi-resolution simulation results can be merged via different toolkits (NCO, GMT, Julia, Python...). We will focus here on Python

In [ ]:
#%%% IMPORT PACKAGES
import xarray as xr
import rioxarray
import numpy as np
import contextily as ctx
import matplotlib.pyplot as plt

#import folium


In [ ]:
#%%% SIMPLE PLOT


# Replace with the path to your output file.
#BGout_path = "\path\to\Tutorial_1\Port_Charles_dx32m.nc"
BGout_path = "/mnt/e/04_BG-Flood/Tutorial/Tutorial_3_Simple_Rain_Port_Charles/Port_Charles_dx32mTo4m.nc"

# Open the dataset with xarray
ds = xr.open_dataset(BGout_path)

# Simple plot
timestep = 4
ds["h_P0"][timestep].plot(vmin=0, vmax=2, cmap="Blues")

# The output is only partial and the variables need to be merged across levels.

In [ ]:
#%%% Merge outputs from different refinement levels.
# The output file is written in the same format as the input file, but with the variables concatenated across levels.
# The output file can be used for further analysis or visualization.

# Function to read and merge BG-Flood output variables variables across levels.
def merge_levels(file_path, list_variables=["h"], list_lvl=[0,1,2], crs="EPSG:2193"):
    # Initialise output file
    dsout = xr.Dataset()
    with xr.open_dataset(BGout_path) as ds:
        for var in list_variables:
            print(f"    . Treatment of variable {var}...")
            for lvl in list_lvl:
                lvl_str = f"P{lvl}"
                da_lvl = ds[f"{var}_{lvl_str}"]
                da_lvl = da_lvl.rename({
                    f"yy_{lvl_str}": "y",
                    f"xx_{lvl_str}": "x",
                })
                da_lvl = da_lvl.rename(var)
                if lvl == maxlevel:
                    da_lvl_out = da_lvl.copy()
                    y = da_lvl_out.y
                    x = da_lvl_out.x
                    continue

                da_lvl = da_lvl.interp(
                    y=y,
                    x=x,
                    method="nearest"
                )

                da_lvl_out = xr.where(
                    da_lvl_out.isnull() & da_lvl.isnull(),
                    np.nan,
                    da_lvl_out.fillna(0) + da_lvl.fillna(0)
                )
                da_lvl_out = da_lvl_out.rio.write_crs("EPSG:2193", inplace=True)

            dsout[var] = da_lvl_out

    dsout.rio.write_crs("EPSG:2193", inplace=True)
    dsout.rio.write_coordinate_system(inplace=True)

    return dsout

BGout_path = "/mnt/e/04_BG-Flood/Tutorial/Tutorial_3_Simple_Rain_Port_Charles/Port_Charles_dx32mTo8m_Kurganov.nc"

# Min and max refinement levels to concatenate.
minlevel = 0
maxlevel = 2

list_lvl = list(range(maxlevel, minlevel-1, -1))

# List of variables to concatenate across levels.
list_variables = ["h", "u", "v", "zs", "hmax", "zsmax", "Umax"]

dsout = merge_levels(BGout_path, list_variables=list_variables, list_lvl=list_lvl, crs="EPSG:2193")
print(" # Reading input file...")


outfile_path = "/mnt/e/04_BG-Flood/Tutorial/Tutorial_3_Simple_Rain_Port_Charles/Port_Charles_dx32mTo8m_Kurganov_concat.nc"
dsout.to_netcdf(outfile_path)


In [36]:
# Sets the level of compression of the file. Try to save with and without compression to see the difference in file size and read/write speed.
encoding = {
    var: {
        "zlib": True,
        "complevel": 6
    }
    for var in dsout.data_vars
}

outfile_path = "/mnt/e/04_BG-Flood/Tutorial/Tutorial_3_Simple_Rain_Port_Charles/Port_Charles_dx32mTo8m_Kurganov_concat_compress.nc"
dsout.to_netcdf(outfile_path, encoding=encoding)